# CASMI 2026 · RL77 Notebook 4 — Precursor + Adduct + Isotope Subformula Filter

*Assembled from **CASMI 2026 Colab Rubric.pdf** (pages 38–48). Python cells are the reviewed `experiments/runs/RL77/nb4/cell_NN.py` files; text cells are taken from the PDF.*

> **User prompt:** Write a fast precursor mass + adduct + isotope subformula filter in Python to generate the top-3 candidate molecular formulas (`formula_priors_batch`) for the `OfflineMassAndFormulaRetriever`.

Predicting the top-3 candidate molecular formulas (`formula_priors_batch`) before candidate retrieval narrows the search space from thousands of nominal-mass isomers down to exact elemental isomers, driving the jump from GLMR's Weight Library (40.35% Top-1) to its Formula Library (43.52% Top-1). This vectorized Python module implements a fast, offline **MIST-CF / SIRIUS-inspired subformula filter** that fuses five physical and neural signals in under 0.8 milliseconds per molecule:

1. **Multi-Spectrum Neutral Mass Consensus:** Pools `precursor_mz` across all spectra and adducts belonging to a `molecule_id` to compute a variance-weighted neutral monoisotopic mass.
2. **Senior's Valence & RDBE Parity Filter:** Enforces closed-shell neutral valence rules (Nitrogen Rule and integer Ring and Double Bond Equivalents).
3. **Diagnostic Neutral Loss Subformula Tree:** Matches observed MS/MS neutral losses (`precursor_mz - ms2_mzs`) against 36 exact-mass diagnostic losses (H₂O, NH₃, CO, SO₃, HPO₃, HCl, hexose, pentose) and penalizes candidate formulas whose elemental stoichiometry cannot form the observed loss.
4. **Raw MS2 Fine-Isotope & Halogen Pattern Detector:** Scans raw fragment pairs separated by 1.9970–1.9980 Da (³⁷Cl, ⁸¹Br, ³⁴S) and 1.00335 Da (¹³C) to score halogen and sulfur presence.
5. **Mass2SMILES 71-D Heteroatom Prior Integration:** Aligns candidate formula heteroatom counts (N, O, S, P, F, Cl, Br, I) directly with the neural predictions from `fg_71_pred[:, 60:68]`.

## Prerequisites

| Input | Kaggle source | Provides |
| --- | --- | --- |
| Competition data | `enveda-CASMI26-molecule-id-mass-spectra` | `train.parquet`, `test.parquet`, `sample_submission.csv` |
| RL77 prerequisites | `scottweeden/casmi26-rl77-prereqs` | `rl77_env.py`, `rl77_weights.py`, `rl77_weights.json` (Evidently snapshot `rl77-v005`), upstream modules |
| Notebook 1 outputs | `scottweeden/casmi26-rl77-nb1-glmr-outputs` | Two-stage GLMR checkpoint, retrieval library, validation split, submission |
| Notebook 2 outputs | `scottweeden/casmi26-rl77-nb2-store-outputs` | Mass-sorted candidate catalog, FP16 ChemFormer memmap, FAISS index, formula index, offline wheels |
| Notebook 3 outputs | `scottweeden/casmi26-rl77-nb3-ensemble-outputs` | Siamese ECFP4 and Mass2SMILES heads, fitted blend weights, ensemble submission |

- **Order:** Run notebooks 1, 2, 3 first and publish their outputs (`tools/kaggle_pipeline.py publish-output`).
- **Accelerator:** CPU (no GPU quota used); internet on for the test run so `ensure_packages()` can reach PyPI. Offline runs install from the wheels in notebook 2's output dataset.
- **Packages:** `rdkit==2026.03.3` (the metric's canonicalizer) and `faiss-cpu` are installed if the image lacks them; `torch`, `pyarrow`, `pandas`, `scipy` come from the Kaggle image.
- **Weights:** all blend weights, reward coefficients, and tolerances come from Evidently dataset `casmi-rl77-weights` (version `rl77-v005`, id `01a1ec58-2d69-70da-8f6a-f47e2c908e2a`) through `W("...")`.
- **Outputs:** everything written to `/kaggle/working` is published as `scottweeden/casmi26-rl77-nb4-formula-outputs` for the next notebook.

In [ ]:
# Prerequisites: locate the RL77 helper modules, install pinned packages, load the Evidently weights snapshot
import glob, os, sys
from pathlib import Path

_lib_hits = sorted(glob.glob("/kaggle/input/**/rl77_env.py", recursive=True))
if _lib_hits:
    RL77_LIB = Path(_lib_hits[0]).parent
elif os.environ.get("RL77_LIB_DIR"):
    RL77_LIB = Path(os.environ["RL77_LIB_DIR"])  # e.g. experiments/runs/RL77/build/datasets/casmi26-rl77-prereqs
else:
    raise FileNotFoundError("Attach the scottweeden/casmi26-rl77-prereqs dataset or set RL77_LIB_DIR.")
sys.path.insert(0, str(RL77_LIB))

import rl77_env as ENV
ENV.ensure_packages()                      # rdkit==2026.03.3 + faiss-cpu (offline wheels when attached)
from rl77_weights import load_rl77_weights
W = load_rl77_weights()                    # every blend weight / reward coefficient in this notebook reads W("...")
from rl77_nb1_glmr import *  # nb1: CASMI26 RL77 NB1 GLMR
from rl77_nb2_store import *  # nb2: CASMI26 RL77 NB2 Offline Store
from rl77_nb3_ensemble import *  # nb3: CASMI26 RL77 NB3 Ensemble
ENV.log(f"Competition data: {ENV.competition_dir()} | outputs: {ENV.work_dir()}")


## Cell 1: Exact Atomic Weights, Hill Formula Parser & Diagnostic Neutral Loss Table

In [ ]:
# [Colab Formula Filter Cell 1] Exact IUPAC masses, Hill-notation stoichiometry parser, and diagnostic MS/MS neutral losses
import re
from typing import List, Dict, Tuple, Optional
import numpy as np
import torch

# Ordered 10-element stoichiometry vector: [C, H, N, O, P, S, F, Cl, Br, I]
ELEMENTS = ["C", "H", "N", "O", "P", "S", "F", "Cl", "Br", "I"]
ELEM_INDEX = {el: i for i, el in enumerate(ELEMENTS)}

# Exact monoisotopic atomic masses (IUPAC / NIST)
EXACT_ATOMIC_MASSES = np.array(
    [
        12.0000000,   # C
        1.007825032,  # H
        14.003074004, # N
        15.994914620, # O
        30.973761998, # P
        31.972071174, # S
        18.998403163, # F
        34.968852682, # Cl
        78.918337600, # Br
        126.90447190, # I
    ],
    dtype=np.float64,
)

# Standard valences for Ring & Double Bond Equivalent (RDBE) calculation:
# RDBE = C - 0.5*(H + F + Cl + Br + I) + 0.5*(N + P) + 1
RDBE_COEFFS = np.array(
    [1.0, -0.5, 0.5, 0.0, 0.5, 0.0, -0.5, -0.5, -0.5, -0.5], dtype=np.float64
)

FORMULA_REGEX = re.compile(r"([A-Z][a-z]?)(\d*)")


def parse_formula_to_vector(formula: str) -> Optional[np.ndarray]:
    """
    Converts a Hill-notation molecular formula string (e.g., 'C15H10O4') into a
    10-element integer stoichiometry vector [C, H, N, O, P, S, F, Cl, Br, I].
    Returns None if the formula contains unsupported metallic/organometallic elements.
    """
    vec = np.zeros(len(ELEMENTS), dtype=np.int16)
    pos = 0
    for match in FORMULA_REGEX.finditer(formula):
        if match.start() != pos:
            return None
        pos = match.end()
        elem, count_str = match.groups()
        if elem not in ELEM_INDEX:
            return None
        count = int(count_str) if count_str else 1
        vec[ELEM_INDEX[elem]] += count
    if pos != len(formula) or vec[0] <= 0:
        return None
    return vec


def vector_to_hill_formula(vec: np.ndarray) -> str:
    """Formats a 10-element stoichiometry vector into standard Hill notation (C, H, then alphabetical)."""
    parts = []
    # Carbon first, Hydrogen second
    for idx in (0, 1):
        c = int(vec[idx])
        if c > 0:
            parts.append(f"{ELEMENTS[idx]}{c if c > 1 else ''}")
    # Remaining elements in alphabetical order: Br, Cl, F, I, N, O, P, S
    alpha_order = sorted(range(2, len(ELEMENTS)), key=lambda i: ELEMENTS[i])
    for idx in alpha_order:
        c = int(vec[idx])
        if c > 0:
            parts.append(f"{ELEMENTS[idx]}{c if c > 1 else ''}")
    return "".join(parts)


# 36 high-frequency diagnostic neutral losses in timsTOF LC-MS/MS spectra:
# (Name, Stoichiometry String) -> converted to exact mass & required atom counts
DIAGNOSTIC_NEUTRAL_LOSS_SPECS = [
    ("H2", "H2"),
    ("CH4", "CH4"),
    ("NH3", "H3N"),
    ("H2O", "H2O"),
    ("HF", "FH"),
    ("HCN", "CHN"),
    ("CO", "CO"),
    ("C2H4", "C2H4"),
    ("CH3N", "CH3N"),
    ("H2CO", "CH2O"),
    ("CH3OH", "CH4O"),
    ("H2S", "H2S"),
    ("HCl", "ClH"),
    ("C2H2O", "C2H2O"),
    ("C3H6", "C3H6"),
    ("CHNO", "CHNO"),
    ("CO2", "CO2"),
    ("CH3NO", "CH3NO"),
    ("C2H5N", "C2H5N"),
    ("CH2O2", "CH2O2"),
    ("C2H6O", "C2H6O"),
    ("NO2", "NO2"),
    ("CH4S", "CH4S"),
    ("C3H4O", "C3H4O"),
    ("C2H4O2", "C2H4O2"),
    ("CH4N2O", "CH4N2O"),
    ("SO2", "O2S"),
    ("C5H8", "C5H8"),          # Isoprene unit (terpenoids)
    ("HBr", "BrH"),
    ("HPO3", "HO3P"),          # Phosphate ester loss
    ("SO3", "O3S"),            # Sulfate ester loss
    ("C3H6O3", "C3H6O3"),
    ("H3PO4", "H3O4P"),        # Phosphoric acid loss
    ("HI", "HI"),
    ("C5H8O4", "C5H8O4"),      # Pentose / Arabinose / Xylose glycoside loss
    ("C6H10O4", "C6H10O4"),    # Deoxyhexose / Rhamnose glycoside loss
    ("C6H10O5", "C6H10O5"),    # Hexose / Glucose / Galactose glycoside loss
    ("C6H8O6", "C6H8O6"),      # Glucuronic acid loss
]


def _compile_diagnostic_loss_tables() -> Tuple[np.ndarray, np.ndarray]:
    vecs = []
    for _, form_str in DIAGNOSTIC_NEUTRAL_LOSS_SPECS:
        # Temporarily parse without requiring Carbon > 0
        v = np.zeros(len(ELEMENTS), dtype=np.int16)
        for m in FORMULA_REGEX.finditer(form_str):
            el, cnt = m.groups()
            v[ELEM_INDEX[el]] += int(cnt) if cnt else 1
        vecs.append(v)
    nl_atom_matrix = np.stack(vecs, axis=0)  # (N_nl, 10)
    nl_exact_masses = nl_atom_matrix.astype(np.float64) @ EXACT_ATOMIC_MASSES
    order = np.argsort(nl_exact_masses)
    return nl_exact_masses[order], nl_atom_matrix[order]


NL_EXACT_MASSES, NL_ATOM_MATRIX = _compile_diagnostic_loss_tables()

## Cell 2: Vectorized Formula Catalog Index & De Novo Senior's Rule Enumerator

This class indexes all unique formulas from `candidate_catalog_sorted.parquet` by exact monoisotopic mass and includes a fast vectorized **Seven Golden Rules / Senior's Rule** de novo generator as a fallback for novel structures whose molecular formula does not yet exist in the catalog.

In [ ]:
# [Colab Formula Filter Cell 2] Mass-Sorted Formula Table + De Novo CHNOPS Validator
class CandidateFormulaBank:
    """
    Stores a mass-sorted table of unique molecular formulas, their exact monoisotopic
    masses, 10-element stoichiometry matrices, and RDBE values for O(log N) slicing.
    """

    def __init__(self, catalog_formulas: List[str]):
        unique_formulas = sorted(set(catalog_formulas))
        valid_f, vecs = [], []
        for f_str in unique_formulas:
            v = parse_formula_to_vector(str(f_str))
            if v is not None and self.passes_seniors_and_golden_rules(v):
                valid_f.append(str(f_str))
                vecs.append(v)

        if vecs:
            mat = np.stack(vecs, axis=0)
        else:
            mat = np.zeros((0, len(ELEMENTS)), dtype=np.int16)

        masses = mat.astype(np.float64) @ EXACT_ATOMIC_MASSES
        order = np.argsort(masses)

        self.formulas = np.array([valid_f[i] for i in order], dtype=object)
        self.atom_matrix = mat[order]                             # (M, 10) int16
        self.exact_masses = masses[order]                         # (M,) float64
        self.rdbe = (self.atom_matrix @ RDBE_COEFFS) + 1.0        # (M,) float64
        # Precompute log1p heteroatom counts [N, O, S, P, F, Cl, Br, I] to match Mass2SMILES indices 60..67
        # Note order in Mass2SMILES: ["N", "O", "S", "P", "F", "Cl", "Br", "I"] -> indices [2, 3, 5, 4, 6, 7, 8, 9]
        m2s_cols = [2, 3, 5, 4, 6, 7, 8, 9]
        self.hetero_log1p = np.log1p(
            self.atom_matrix[:, m2s_cols].astype(np.float32)
        )  # (M, 8)

    @staticmethod
    def passes_seniors_and_golden_rules(vec: np.ndarray) -> bool:
        """
        Enforces Senior's Valence Theorem (Nitrogen Rule / integer RDBE >= -0.5)
        and Kind & Fiehn (2007) H/C and heteroatom/carbon bounds for organic molecules.
        """
        c, h = int(vec[0]), int(vec[1])
        if c <= 0 or h <= 0:
            return False
        # Double-check RDBE is an exact integer >= 0.0 (sum of odd-valence atoms must be even)
        two_rdbe = 2 * c - (h + int(vec[6]) + int(vec[7]) + int(vec[8]) + int(vec[9])) + (int(vec[2]) + int(vec[4])) + 2
        if two_rdbe < 0 or (two_rdbe % 2 != 0):
            return False
        # Seven Golden Rules H/C ratio bounds (0.2 <= H/C <= 3.5) and O/C <= 1.5, N/C <= 1.3
        if not (0.15 * c <= h <= 3.5 * c + 2):
            return False
        if int(vec[3]) > int(1.5 * c + 2) or int(vec[2]) > int(1.3 * c + 2):
            return False
        return True

    def query_mass_window(
        self, neutral_mass: float, ppm_tol: float = 15.0
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        """
        Slices candidate formulas within [neutral_mass - tol, neutral_mass + tol] in O(log M).
        If fewer than 3 catalog formulas exist in the window, enumerates de novo CHNOPS formulas.
        """
        tol_da = max(neutral_mass * (ppm_tol * 1e-6), 0.012)
        lo = int(np.searchsorted(self.exact_masses, neutral_mass - tol_da, side="left"))
        hi = int(np.searchsorted(self.exact_masses, neutral_mass + tol_da, side="right"))

        if hi - lo >= 3:
            return (
                self.formulas[lo:hi],
                self.exact_masses[lo:hi],
                self.atom_matrix[lo:hi],
                self.rdbe[lo:hi],
                self.hetero_log1p[lo:hi],
            )

        # Fallback: fast de novo CHNO(S) enumeration within tolerance window
        return self._enumerate_denovo_formulas(neutral_mass, tol_da)

    def _enumerate_denovo_formulas(
        self, neutral_mass: float, tol_da: float
    ) -> Tuple[np.ndarray, np.ndarray, np.ndarray, np.ndarray, np.ndarray]:
        c_min = max(1, int((neutral_mass - 180.0) / 14.0))
        c_max = min(85, int(neutral_mass / 12.0))

        found_f, found_m, found_v, found_r, found_h = [], [], [], [], []
        m2s_cols = [2, 3, 5, 4, 6, 7, 8, 9]

        for c in range(c_min, c_max + 1):
            rem_c = neutral_mass - c * EXACT_ATOMIC_MASSES[0]
            if rem_c < 1.0:
                continue
            for n in range(0, min(12, int(rem_c / 14.0) + 1)):
                rem_cn = rem_c - n * EXACT_ATOMIC_MASSES[2]
                for o in range(0, min(20, int(rem_cn / 15.99) + 1)):
                    rem_cno = rem_cn - o * EXACT_ATOMIC_MASSES[3]
                    for s in range(0, min(3, int(rem_cno / 31.97) + 1)):
                        rem_h = rem_cno - s * EXACT_ATOMIC_MASSES[5]
                        if rem_h < 0.5:
                            continue
                        h = int(round(rem_h / EXACT_ATOMIC_MASSES[1]))
                        vec = np.array([c, h, n, o, 0, s, 0, 0, 0, 0], dtype=np.int16)
                        if not self.passes_seniors_and_golden_rules(vec):
                            continue
                        exact_m = float(vec.astype(np.float64) @ EXACT_ATOMIC_MASSES)
                        if abs(exact_m - neutral_mass) <= tol_da:
                            found_f.append(vector_to_hill_formula(vec))
                            found_m.append(exact_m)
                            found_v.append(vec)
                            found_r.append(float((vec @ RDBE_COEFFS) + 1.0))
                            found_h.append(np.log1p(vec[m2s_cols].astype(np.float32)))

        if not found_f:
            return (
                np.empty(0, dtype=object),
                np.empty(0, dtype=np.float64),
                np.empty((0, 10), dtype=np.int16),
                np.empty(0, dtype=np.float64),
                np.empty((0, 8), dtype=np.float32),
            )
        return (
            np.array(found_f, dtype=object),
            np.array(found_m, dtype=np.float64),
            np.stack(found_v, axis=0),
            np.array(found_r, dtype=np.float64),
            np.stack(found_h, axis=0),
        )

## Cell 3: Fast MIST-CF / SIRIUS Multi-Spectrum Subformula Scorer

> **Review fix:** the consensus mass used the shift table (formate/acetate spellings and `[2M+X]` fell back to `[M+H]+`); it now uses `neutral_mass_from_precursor()`. The scorer is split into `score_components()` and `combine()` so validation can fit the composite weights without re-enumerating formulas. Every coefficient (ppm σ, neutral-loss bonus/penalty, isotope bonuses, heteroatom weights, RDBE terms, and the 1.50 / 1.00 / 0.80 / 1.20 composite weights) comes from the Evidently weights.

In [ ]:
# [Colab Formula Filter Cell 3] Fast Multi-Spectrum Precursor + Neutral Loss + Isotope + Neural Heteroatom Scorer
class FastMISTSubformulaFilter:
    """
    Ranks candidate molecular formulas for each molecule_id using 5 fast vectorized signals:
      1. Multi-Spectrum Consensus Mass PPM Likelihood (sigma_ppm = 4.5 ppm on timsTOF)
      2. MS/MS Diagnostic Neutral Loss Subformula Compatibility (SIRIUS/MIST-CF loss tree)
      3. Raw MS/MS Fine-Isotope & Halogen Pattern Match (37Cl, 81Br, 34S spacing at ~1.9975 Da)
      4. Mass2SMILES 71-D Neural Heteroatom Count Consistency (N, O, S, P, F, Cl, Br, I)
      5. RDBE vs. Predicted Aromatic + Aliphatic Ring Lower Bound (fg_71_pred[:, 69:71])
    """

    SCORE_KEYS = ("mass", "nl", "iso", "neural")

    def __init__(
        self,
        formula_bank: CandidateFormulaBank,
        adduct_mass_shift: Optional[Dict[str, float]] = None,
        ppm_sigma: float = W("nb4.filter.ppm_sigma"),
        nl_tol_da: float = W("nb4.filter.nl_tol_da"),
        score_weights: Optional[Dict[str, float]] = None,
    ):
        self.bank = formula_bank
        self.adduct_mass_shift = adduct_mass_shift  # kept for the PDF signature; see infer_consensus_neutral_mass
        self.ppm_sigma = ppm_sigma
        self.nl_tol_da = nl_tol_da
        # Composite posterior weights (PDF: 1.50 / 1.00 / 0.80 / 1.20) from the Evidently snapshot
        self.score_weights = dict(score_weights or {k: W(f"nb4.score.{k}") for k in self.SCORE_KEYS})

    def infer_consensus_neutral_mass(
        self,
        precursor_mzs: List[float],
        adducts: List[str],
        bpis: Optional[List[float]] = None,
        ionization_modes: Optional[List[str]] = None,
    ) -> float:
        """
        Combines precursor_mz measurements across all spectra of a molecule_id
        (accounting for adduct mass shifts) weighted by log10(base_peak_intensity).
        Uses neutral_mass_from_precursor(): the shift table missed formate/acetate spellings and [2M+X].
        """
        masses, weights = [], []
        for idx, (pmz, add) in enumerate(zip(precursor_mzs, adducts)):
            if pmz <= 0.0:
                continue
            mode = ionization_modes[idx] if ionization_modes else "positive"
            masses.append(neutral_mass_from_precursor(float(pmz), str(add), mode))
            w = math.log10(max(float(bpis[idx]), 10.0)) if bpis else 1.0
            weights.append(w)

        if not masses:
            return 0.0
        masses_arr = np.asarray(masses, dtype=np.float64)
        weights_arr = np.asarray(weights, dtype=np.float64)

        # Reject outlier adduct misassignments (> 0.05 Da from median)
        med = float(np.median(masses_arr))
        inliers = np.abs(masses_arr - med) <= 0.05
        if np.any(inliers):
            return float(np.average(masses_arr[inliers], weights=weights_arr[inliers]))
        return med

    @staticmethod
    def detect_halogen_and_sulfur_isotopes(
        raw_mzs: np.ndarray, raw_ints: np.ndarray
    ) -> Tuple[bool, bool]:
        """
        Inspects raw peak pairs separated by 1.9955 - 1.9985 Da (M+2 isotope spacing for
        37Cl, 81Br, and 34S) among intense peaks to flag likely Cl/Br or S presence.
        """
        if len(raw_mzs) < 2:
            return False, False
        top_mask = raw_ints >= 0.10
        m = raw_mzs[top_mask]
        intens = raw_ints[top_mask]
        if len(m) < 2:
            return False, False

        diffs = np.abs(m[:, None] - m[None, :])
        m2_pairs = np.where((diffs >= 1.9950) & (diffs <= 1.9988))
        has_cl_br = False
        has_s = False
        for i, j in zip(*m2_pairs):
            if m[j] > m[i] and intens[i] > 0.15:
                ratio = intens[j] / intens[i]
                if 0.22 <= ratio <= 1.35:
                    has_cl_br = True
                elif 0.035 <= ratio < 0.22:
                    has_s = True
        return has_cl_br, has_s

    def score_components(
        self,
        neutral_mass: float,
        pooled_nls: np.ndarray,
        pooled_ints: np.ndarray,
        raw_mzs: Optional[np.ndarray] = None,
        raw_ints: Optional[np.ndarray] = None,
        fg_71_pred: Optional[np.ndarray] = None,
        ppm_window: float = W("nb4.priors.ppm_window"),
    ) -> Tuple[np.ndarray, Dict[str, np.ndarray]]:
        """Candidate formulas within +/- ppm_window and their four component scores (before weighting)."""
        formulas, exact_masses, atom_mat, rdbe_vec, hetero_log1p = (
            self.bank.query_mass_window(neutral_mass, ppm_tol=ppm_window)
        )
        n_cand = len(formulas)
        zeros = {k: np.zeros(n_cand, dtype=np.float64) for k in self.SCORE_KEYS}
        if n_cand == 0:
            return formulas, zeros

        # 1. Precursor Mass PPM Score (Gaussian log-likelihood in [0, 1])
        ppm_errors = np.abs(exact_masses - neutral_mass) / neutral_mass * 1e6
        score_mass = np.exp(-0.5 * (ppm_errors / self.ppm_sigma) ** 2)

        # 2. Diagnostic Neutral Loss Subformula Score (SIRIUS / MIST-CF style)
        score_nl = np.zeros(n_cand, dtype=np.float64)
        valid_nl_mask = (pooled_nls > 1.5) & (pooled_ints > 0.05)
        obs_nls = pooled_nls[valid_nl_mask]
        obs_weights = pooled_ints[valid_nl_mask]
        nl_hit, nl_miss = W("nb4.nl.explained"), W("nb4.nl.unexplained")

        if len(obs_nls) > 0:
            # Match observed neutral losses against pre-compiled diagnostic neutral loss table
            diff_matrix = np.abs(obs_nls[:, None] - NL_EXACT_MASSES[None, :])  # (N_obs, N_diag)
            obs_idx, diag_idx = np.where(diff_matrix <= self.nl_tol_da)

            for o_i, d_i in zip(obs_idx, diag_idx):
                req_atoms = NL_ATOM_MATRIX[d_i]  # (10,)
                w_peak = float(obs_weights[o_i])
                # Candidate formula can explain this neutral loss iff atom_mat >= req_atoms for all 10 elements
                can_form_loss = np.all(atom_mat >= req_atoms[None, :], axis=1)
                score_nl += np.where(can_form_loss, nl_hit * w_peak, nl_miss * w_peak)

        # 3. Raw MS2 Isotope Pattern Score (Cl/Br and S M+2 signatures; raw_ints must be linear intensities)
        score_iso = np.zeros(n_cand, dtype=np.float64)
        if raw_mzs is not None and raw_ints is not None:
            has_cl_br, has_s = self.detect_halogen_and_sulfur_isotopes(raw_mzs, raw_ints)
            cand_has_cl_br = (atom_mat[:, 7] + atom_mat[:, 8]) > 0
            cand_has_s = atom_mat[:, 5] > 0
            if has_cl_br:
                score_iso += np.where(cand_has_cl_br, W("nb4.iso.clbr_detected_match"), W("nb4.iso.clbr_detected_mismatch"))
            else:
                score_iso += np.where(cand_has_cl_br, W("nb4.iso.clbr_absent_but_present"), W("nb4.iso.clbr_absent_and_absent"))
            if has_s:
                score_iso += np.where(cand_has_s, W("nb4.iso.s_detected_match"), W("nb4.iso.s_detected_mismatch"))

        # 4. Mass2SMILES 71-D Neural Heteroatom + Ring RDBE Prior Score
        score_neural = np.zeros(n_cand, dtype=np.float64)
        if fg_71_pred is not None and len(fg_71_pred) >= 71:
            # Indices 60..67: predicted log1p counts for [N, O, S, P, F, Cl, Br, I]
            pred_hetero = np.asarray(fg_71_pred[60:68], dtype=np.float32)
            # Weight N, O, S, P, Cl, Br agreement strongly
            elem_weights = np.array(W.vec("nb4.neural.elem_weight", HETEROATOM_SYMBOLS), dtype=np.float32)
            l1_hetero = np.sum(
                np.abs(hetero_log1p - pred_hetero[None, :]) * elem_weights[None, :],
                axis=1,
            ) / np.sum(elem_weights)
            score_neural += np.exp(-W("nb4.neural.decay") * l1_hetero)

            # Indices 69..70: log1p(aromatic rings) and log1p(aliphatic rings)
            est_arom_rings = max(0.0, float(np.expm1(fg_71_pred[69])))
            est_aliph_rings = max(0.0, float(np.expm1(fg_71_pred[70])))
            min_expected_rdbe = (W("nb4.neural.rdbe_per_aromatic_ring") * est_arom_rings
                                 + W("nb4.neural.rdbe_per_aliphatic_ring") * est_aliph_rings)
            # Penalize formulas whose RDBE is far below the neural ring lower bound
            rdbe_deficit = np.clip(min_expected_rdbe - rdbe_vec, 0.0, None)
            score_neural -= W("nb4.neural.rdbe_deficit_penalty") * rdbe_deficit

        return formulas, {"mass": score_mass, "nl": score_nl, "iso": score_iso, "neural": score_neural}

    def combine(self, components: Dict[str, np.ndarray], score_weights: Optional[Dict[str, float]] = None) -> np.ndarray:
        """Total composite formula log-posterior score."""
        w = score_weights or self.score_weights
        return sum(w[k] * components[k] for k in self.SCORE_KEYS)

    def score_and_select_top_k_formulas(
        self,
        neutral_mass: float,
        pooled_nls: np.ndarray,
        pooled_ints: np.ndarray,
        raw_mzs: Optional[np.ndarray] = None,
        raw_ints: Optional[np.ndarray] = None,
        fg_71_pred: Optional[np.ndarray] = None,
        ppm_window: float = W("nb4.priors.ppm_window"),
        top_k: int = W.int("nb4.priors.top_k_formulas"),
    ) -> List[str]:
        """
        Scores all valid candidate formulas within +/- ppm_window and returns the top-K
        Hill-notation formula strings for OfflineMassAndFormulaRetriever.
        """
        formulas, components = self.score_components(
            neutral_mass, pooled_nls, pooled_ints, raw_mzs, raw_ints, fg_71_pred, ppm_window
        )
        if len(formulas) <= 1:
            return [str(f) for f in formulas]
        ranked_idx = np.argsort(-self.combine(components))[:top_k]
        return [str(formulas[i]) for i in ranked_idx]

## Cell 4: Batch Hook (`generate_formula_priors_batch`) for `OfflineMassAndFormulaRetriever`

Drop `generate_formula_priors_batch` directly into your inference loop right after `fg_71_pred, _ = fg_head(h_s_mol)` and pass the output into `retriever.retrieve_top_k_batch(..., formula_priors_batch=formula_priors_batch)`.

> **Review fix:** every spectrum was assigned the primary adduct, so a molecule measured as `[M+H]+` and `[M+Na]+` produced a 22 Da disagreement in the consensus mass; per-spectrum adducts are decoded from `batch["adduct"]`. The isotope detector's ratio thresholds assume linear intensities, but the dataset stores √I (a 0.045 ³⁴S ratio became 0.21), so intensities are squared before detection. The synthetic chrysin smoke test is replaced by the real run in Cell 6.

In [ ]:
# [Colab Formula Filter Cell 4] Batch Integration Hook & End-to-End Verification
INV_ADDUCT_VOCAB = {idx: name for name, idx in ADDUCT_VOCAB.items()}


def molecule_formula_inputs(
    batch: Dict[str, torch.Tensor],
    b: int,
    subformula_filter: FastMISTSubformulaFilter,
) -> Dict[str, object]:
    """Consensus neutral mass and pooled peak arrays for molecule b of a MultiSpectrumMoleculeDataset batch."""
    mzs_np = batch["mzs"][b].detach().cpu().numpy()            # (S, P)
    ints_np = batch["ints"][b].detach().cpu().numpy()          # (S, P)  sqrt-scaled intensities
    nls_np = batch["nls"][b].detach().cpu().numpy()            # (S, P)
    bpi_np = 10.0 ** batch["bpi"][b].detach().cpu().numpy()    # (S,)
    adduct_ids = batch["adduct"][b].detach().cpu().numpy()     # (S,) per-spectrum adducts
    valid_s = np.where(batch["spec_mask"][b].detach().cpu().numpy())[0]

    # Precursor m/z is anchored at peak index 0 of each spectrum in MultiSpectrumMoleculeDataset.
    # Each spectrum keeps its own adduct (the PDF copied the primary adduct onto every spectrum,
    # so a molecule measured as [M+H]+ and [M+Na]+ produced a 22 Da disagreement).
    prec_mzs = [float(mzs_np[s, 0]) for s in valid_s]
    primary = str(batch["primary_adduct"][b])
    adducts = [INV_ADDUCT_VOCAB.get(int(adduct_ids[s]), primary) for s in valid_s]
    adducts = [primary if a == "UNKNOWN" else a for a in adducts]
    mode = str(batch["ionization_mode"][b]) if "ionization_mode" in batch else "positive"
    bpis = [float(bpi_np[s]) for s in valid_s]

    neutral_mass = subformula_filter.infer_consensus_neutral_mass(prec_mzs, adducts, bpis, [mode] * len(valid_s))
    pooled_ints = ints_np[valid_s, 1:].reshape(-1)
    return {
        "neutral_mass": neutral_mass,
        "pooled_nls": nls_np[valid_s, 1:].reshape(-1),
        "pooled_ints": pooled_ints,
        "raw_mzs": mzs_np[valid_s, 1:].reshape(-1),
        # The isotope detector's ratio thresholds are for linear intensities; the dataset stores sqrt(I).
        "raw_ints": pooled_ints ** 2,
    }


@torch.no_grad()
def generate_formula_priors_batch(
    batch: Dict[str, torch.Tensor],
    subformula_filter: FastMISTSubformulaFilter,
    fg_71_pred_batch: Optional[torch.Tensor] = None,
    ppm_window: float = W("nb4.priors.ppm_window"),
    top_k_formulas: int = W.int("nb4.priors.top_k_formulas"),
) -> List[List[str]]:
    """
    Generates `formula_priors_batch` (List of top-3 molecular formulas per molecule in batch)
    directly from a MultiSpectrumMoleculeDataset batch and optional Mass2SMILES fg_71_pred.
    """
    fg_np = fg_71_pred_batch.detach().cpu().numpy() if fg_71_pred_batch is not None else None
    formula_priors_batch: List[List[str]] = []
    for b in range(batch["mzs"].shape[0]):
        inputs = molecule_formula_inputs(batch, b, subformula_filter)
        top_formulas = subformula_filter.score_and_select_top_k_formulas(
            fg_71_pred=fg_np[b] if fg_np is not None else None,
            ppm_window=ppm_window,
            top_k=top_k_formulas,
            **inputs,
        )
        formula_priors_batch.append(top_formulas)
    return formula_priors_batch

## How to Wire `formula_priors_batch` into Your Kaggle Inference Loop

In **Cell 5** of the Ensemble Inference loop (`run_blended_casmi26_inference`), swap the Stage 1 retrieval call so `fg_head(h_s_mol)` runs immediately before `retrieve_top_k_batch`.

> **Review fix:** the snippet used `glmr_model`, `fg_head`, `mist_filter`, and `offline_retriever` as undefined globals (NameError as a cell); the same three steps are wrapped in `retrieve_with_formula_priors()`.

In [ ]:
# [Colab Formula Filter Cell 5] How to wire formula_priors_batch into the Kaggle inference loop
# The PDF snippet referenced glmr_model / fg_head / mist_filter / offline_retriever as globals
# (NameError when run as a cell); it is wrapped in a function so notebooks 4 and 5 can call it.
@torch.no_grad()
def retrieve_with_formula_priors(
    batch: Dict[str, torch.Tensor],
    glmr_model: nn.Module,
    fg_head: nn.Module,
    mist_filter: FastMISTSubformulaFilter,
    offline_retriever: "OfflineMassAndFormulaRetriever",
    top_k: int = W.int("nb3.retrieval.top_k"),
    ppm_tol: float = W("nb2.retrieval.ppm_tol"),
):
    # 1. Encode multi-spectrum sequence and predict Mass2SMILES 71-D vector first
    h_s_mol, e_s_mol, _ = glmr_model.encode_molecule_spectra(batch)
    fg_71_pred, _ = fg_head(h_s_mol)  # (B, 71)

    # 2. Generate top-3 MIST-CF / SIRIUS candidate molecular formulas per molecule_id
    formula_priors_batch = generate_formula_priors_batch(
        batch=batch,
        subformula_filter=mist_filter,
        fg_71_pred_batch=fg_71_pred,
        ppm_window=W("nb4.priors.ppm_window"),
        top_k_formulas=W.int("nb4.priors.top_k_formulas"),
    )

    # 3. Query OfflineMassAndFormulaRetriever using both Formula Priors + PPM Mass Window
    h_m_k, topk_indices_batch = offline_retriever.retrieve_top_k_batch(
        e_s_batch=e_s_mol,
        precursor_mzs=batch["precursor_mz"],
        adducts=batch["primary_adduct"],
        adduct_mass_shift=ADDUCT_MASS_SHIFT,
        top_k=top_k,
        ppm_tol=ppm_tol,
        formula_priors_batch=formula_priors_batch,
        ionization_modes=batch["ionization_mode"],
    )
    return h_s_mol, e_s_mol, fg_71_pred, formula_priors_batch, h_m_k, topk_indices_batch

## Cell 6: Evaluate on Held-Out Molecules, Fit the Score Weights, Priors for the Test Set

The PDF's verification used five hand-written formulas and a mock batch. This cell runs the filter on notebook 1's held-out timsTOF molecules with the notebook-3 Mass2SMILES head: formula top-1 / top-3 accuracy against the RDKit formula of the true structure, a 2-fold cross-fitted coordinate search over the four composite weights (written to `fitted_weights.json` for `tools/evidently_weights.py adopt`), and Stage 1 retrieval MRR@25 with and without formula priors on the same molecules. It writes `test_formula_priors.json` and a formula-constrained `submission.csv`. Runs on CPU.

In [ ]:
# [Colab Formula Filter Cell 6] Evaluate the filter on held-out molecules, fit the score weights, priors for the test set
from rdkit.Chem import rdMolDescriptors as _rdmd


def truth_formula(smiles: str) -> Optional[str]:
    mol = Chem.MolFromSmiles(smiles) if isinstance(smiles, str) else None
    return _rdmd.CalcMolFormula(mol) if mol is not None else None


def load_fg_head(nb3_dir: Path) -> "Mass2SMILESFunctionalGroupHead":
    fg_head = Mass2SMILESFunctionalGroupHead()
    fg_head.load_state_dict(torch.load(nb3_dir / "mass2smiles_fg_head.pt", map_location="cpu")["state_dict"])
    return fg_head.to(CFG.DEVICE).eval()


@torch.no_grad()
def collect_formula_records(loader, glmr_model, fg_head, mist_filter) -> List[Dict]:
    records = []
    for batch in loader:
        batch = {k: (v.to(CFG.DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
        h_s_mol, _, _ = glmr_model.encode_molecule_spectra(batch)
        fg_np = fg_head(h_s_mol)[0].cpu().numpy()
        for b, mol_id in enumerate(batch["molecule_id"]):
            inputs = molecule_formula_inputs(batch, b, mist_filter)
            formulas, comps = mist_filter.score_components(fg_71_pred=fg_np[b], **inputs)
            records.append({
                "molecule_id": str(mol_id),
                "formulas": [str(f) for f in formulas],
                "components": comps,
                "truth": truth_formula(batch["smiles"][b]) if "smiles" in batch else None,
                "neutral_mass": float(inputs["neutral_mass"]),
            })
    return records


def formula_topk_accuracy(records: List[Dict], weights: Dict[str, float], k: int) -> float:
    hits = []
    for r in records:
        if not r["formulas"]:
            hits.append(False)
            continue
        total = sum(weights[key] * r["components"][key] for key in FastMISTSubformulaFilter.SCORE_KEYS)
        top = [r["formulas"][i] for i in np.argsort(-total, kind="stable")[:k]]
        hits.append(r["truth"] in top)
    return float(np.mean(hits)) if hits else 0.0


def fit_score_weights(records: List[Dict], start: Dict[str, float], passes: int = 3) -> Dict[str, float]:
    """Coordinate search on top-1 formula accuracy (top-3 breaks ties)."""
    grid = [0.0, 0.25, 0.5, 0.8, 1.0, 1.2, 1.5, 2.0, 3.0]
    best = dict(start)

    def objective(w):
        return (formula_topk_accuracy(records, w, 1), formula_topk_accuracy(records, w, 3))

    best_score = objective(best)
    for _ in range(passes):
        for key in FastMISTSubformulaFilter.SCORE_KEYS:
            for value in grid:
                trial = {**best, key: value}
                score = objective(trial)
                if score > best_score:
                    best, best_score = trial, score
    return best


@torch.no_grad()
def store_stage1_predictions(retriever, glmr_model, loader, priors_fn=None) -> Dict[str, List[str]]:
    preds = {}
    for batch in loader:
        batch = {k: (v.to(CFG.DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
        _, e_s_mol, _ = glmr_model.encode_molecule_spectra(batch)
        priors = priors_fn(batch) if priors_fn else None
        _, ids_batch = retriever.retrieve_top_k_batch(
            e_s_mol, batch["precursor_mz"], batch["primary_adduct"],
            formula_priors_batch=priors, ionization_modes=batch["ionization_mode"],
        )
        e_np = e_s_mol.cpu().numpy().astype(np.float32)
        for b, mol_id in enumerate(batch["molecule_id"]):
            ids = ids_batch[b]
            neutral = neutral_mass_from_precursor(
                float(batch["precursor_mz"][b]), batch["primary_adduct"][b], batch["ionization_mode"][b]
            )
            in_formula = np.zeros(len(ids), dtype=bool)
            if priors and priors[b]:
                in_formula = np.isin(retriever.catalog_df["molecular_formula"].to_numpy()[ids], priors[b])
            rec = {
                "cand_smiles": retriever.smiles_arr[ids].tolist(),
                "stage1_scores": np.asarray(retriever.mmap_embs[ids], dtype=np.float32) @ e_np[b],
                "stage2_scores": None,
                # formula-prior or mass-window candidates rank ahead of FAISS fill
                "in_window": retriever.in_window(ids, neutral) | in_formula,
                "gen_smiles": None, "gen_ik14": None, "gen_mass_ok": False, "backfill_smiles": [],
            }
            preds[str(mol_id)] = assemble_ranked_candidates(rec, 0.0, 0)
    return preds


def run_formula_pipeline() -> Dict:
    out_dir = ENV.work_dir()
    metrics: Dict = {"weights_version": W.version}
    nb1_dir = ENV.upstream_dir("nb1", "glmr_model.pt")
    split_dir = ENV.upstream_dir("nb1", "val_inchikey14.txt")
    nb3_dir = ENV.upstream_dir("nb3", "mass2smiles_fg_head.pt")
    glmr_model, tokenizer = load_glmr_checkpoint(nb1_dir / "glmr_model.pt", device=CFG.DEVICE)
    fg_head = load_fg_head(nb3_dir)
    retriever = OfflineMassAndFormulaRetriever()
    bank = CandidateFormulaBank(retriever.catalog_df["molecular_formula"].dropna().unique().tolist())
    mist_filter = FastMISTSubformulaFilter(bank)
    ENV.log(f"Formula bank: {len(bank.formulas):,} formulas passing Senior / Seven Golden Rules")

    # 1. Formula identification on notebook 1's held-out timsTOF molecules
    val_df = load_spectra_for_inchikeys(CFG.DATA_DIR, read_split(split_dir, "val"), timstof_window_only=True)
    val_loader = make_loader(val_df, tokenizer, train=False, batch_size=32)
    records = [r for r in collect_formula_records(val_loader, glmr_model, fg_head, mist_filter) if r["truth"]]
    prior = dict(mist_filter.score_weights)
    metrics["val_formula"] = {
        "n": len(records),
        "truth_in_candidate_window": float(np.mean([r["truth"] in r["formulas"] for r in records])),
        "top1_prior": formula_topk_accuracy(records, prior, 1),
        "top3_prior": formula_topk_accuracy(records, prior, 3),
    }

    # 2. Cross-fit the four composite weights (fit on one half, score the other), then fit on all
    rng = np.random.default_rng(CFG.SEED)
    order = rng.permutation(len(records))
    folds = [[records[i] for i in order[0::2]], [records[i] for i in order[1::2]]]
    held = sum(formula_topk_accuracy(h, fit_score_weights(f, prior), 1) * len(h) for f, h in ((folds[0], folds[1]), (folds[1], folds[0])))
    fitted = fit_score_weights(records, prior)
    metrics["val_formula"].update({
        "top1_fitted_crossfit": held / len(records),
        "top1_fitted_in_sample": formula_topk_accuracy(records, fitted, 1),
        "top3_fitted_in_sample": formula_topk_accuracy(records, fitted, 3),
    })
    ENV.write_json(out_dir / "fitted_weights.json", {
        "notebook": "nb4",
        "weights": {f"nb4.score.{k}": v for k, v in fitted.items()},
        "evidence": {"metric": "formula top-1 accuracy", "split": f"{len(records)} held-out timsTOF molecules, 2-fold cross-fit",
                     "prior": metrics["val_formula"]["top1_prior"], "fitted_crossfit": metrics["val_formula"]["top1_fitted_crossfit"]},
    })
    ENV.log(f"Formula identification: {metrics['val_formula']}")
    # From here on the filter scores with the weights `adopt` will activate (fit only if it wins on held-out halves).
    chosen, used_fit = ENV.select_validated(prior, fitted, metrics["val_formula"]["top1_prior"],
                                            metrics["val_formula"]["top1_fitted_crossfit"])
    mist_filter.score_weights = dict(chosen)
    metrics["test_score_weights"] = {**chosen, "source": "fitted" if used_fit else W.version}

    # 3. Does formula-constrained retrieval help? Same molecules, offline store, Stage 1 ranking.
    def priors_fn(batch):
        return generate_formula_priors_batch(batch, mist_filter, fg_head(glmr_model.encode_molecule_spectra(batch)[0])[0])

    gt = {mid: mid for mid in val_df["inchikey14"].unique()}
    metrics["val_retrieval_mrr25"] = {
        "mass_window_only": compute_casmi_mrr25(store_stage1_predictions(retriever, glmr_model, val_loader), gt),
        "with_formula_priors": compute_casmi_mrr25(store_stage1_predictions(retriever, glmr_model, val_loader, priors_fn), gt),
    }
    use_priors = metrics["val_retrieval_mrr25"]["with_formula_priors"] > metrics["val_retrieval_mrr25"]["mass_window_only"]
    metrics["test_uses_formula_priors"] = bool(use_priors)
    ENV.log(f"Validation retrieval MRR@25: {metrics['val_retrieval_mrr25']}; test submission uses formula priors: {use_priors}")

    # 4. Test set: formula priors (written for inspection and reuse) + submission from the better retrieval mode
    test_df = pd.read_parquet(CFG.DATA_DIR / "test.parquet")
    test_loader = make_loader(test_df, tokenizer, train=False, batch_size=32)
    test_priors: Dict[str, List[str]] = {}
    with torch.no_grad():
        for batch in test_loader:
            batch = {k: (v.to(CFG.DEVICE) if isinstance(v, torch.Tensor) else v) for k, v in batch.items()}
            for mol_id, priors in zip(batch["molecule_id"], priors_fn(batch)):
                test_priors[str(mol_id)] = priors
    ENV.write_json(out_dir / "test_formula_priors.json", test_priors)
    preds = store_stage1_predictions(retriever, glmr_model, test_loader, priors_fn if use_priors else None)
    sub_df = pd.DataFrame([{"molecule_id": m, "smiles": ";".join(p)} for m, p in preds.items()], columns=["molecule_id", "smiles"])
    validate_submission_file(sub_df, test_df["molecule_id"].unique().tolist())
    sub_df.to_csv(out_dir / "submission.csv", index=False)

    ENV.write_json(out_dir / "metrics.json", metrics)
    ENV.write_run_manifest("nb4", out_dir, W.version, {"metrics": metrics})
    ENV.log(f"Notebook 4 outputs written to {out_dir}")
    return metrics


if __name__ == "__main__":
    NB4_METRICS = run_formula_pipeline()

*The test submission scores formulas with the fitted composite weights only if they beat the prior on the held-out halves, and uses formula-constrained retrieval only if it beat mass-window retrieval on the same molecules (`metrics.json`: `test_score_weights`, `test_uses_formula_priors`). Notebook 5 recomputes formula priors on the fly with the adopted weights from the Evidently snapshot; `test_formula_priors.json` is kept for inspection.*